# About

This notebook can train a model ("In-notebook training") and run a generate (predict) task given a trained model ("In-notebook generation").

This notebook can be used to interrogate patient histories at the individual ("Patient-level analysis") or population level ("Population-level analysis").

# Setup

## Directories
Relative to current

In [ ]:
_datapath = "./test/data"
_checkpointpath = "./checkpoints"
_configpath = "./configs"

## Python imports

In [ ]:
%load_ext autoreload
%autoreload 2
import difflib
import gzip
import json
import os
import pickle
from collections import Counter, defaultdict
from datetime import date
from glob import glob

import matplotlib._color_data as mcd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import zscore
from tensorboard.backend.event_processing.event_file_loader import EventFileLoader
from tqdm import tqdm
from patient_tpp.analysis import patient_piechart, patient_timeline, contains_coi_not_in_prelookback, rmse
from patient_tpp.basics import is_outcome_of_interest

from patient_tpp.easy_tpp_zai.config_factory.config import ZaiConfig
from patient_tpp.easy_tpp_zai.runner.tpp_runner import ZaiTPPRunner

cwd = os.getcwd()
_datadir = os.path.join(cwd, _datapath)
_checkpoint_dir = os.path.join(cwd, _checkpointpath)
_config_dir = os.path.join(cwd, _configpath)

# Load assets

### event types modeled in manuscript

In [ ]:
with open(os.path.join(_datadir, "enc2thingnamerval.pkl"), "rb") as fobj:
    enc2thingnamerval = pickle.load(fobj)

In [ ]:
thingnamerval2enc = {v: k for k, v in enc2thingnamerval.items()}
colnames = [enc2thingnamerval[k] for k in sorted(enc2thingnamerval.keys())]

In [ ]:
thingnames = set()
for k in thingnamerval2enc.keys():
    thingnames.add(k.split("^")[0])
len(thingnames)

In [ ]:
arranged_thingnames = sorted(
    [(thingname, is_outcome_of_interest(thingname)) for thingname in thingnames],
    key=lambda v: (-v[1], v[0])
)
enc2thingname = {i: v[0] for i, v in enumerate(arranged_thingnames)}
thingname2enc = {v: k for k, v in enc2thingname.items()}
enc2thingname[65] = "padding token"

# In-notebook training
Notes:   
initializing the config creates the directory where the model will ultimately be stored. Example- ./checkpoints/961_140390730688320_260108-050727  
  
Training takes on the order of hours

In [ ]:
config = ZaiConfig.build_from_yaml_file(os.path.join(_config_dir, "config.yaml"), experiment_id="AttNHP_zai_train")

In [ ]:
model_runner = ZaiTPPRunner.build_from_config(config)
model_runner.run()

# In-notebook generation
Note:  
if you would like to utilize a model that you just trained, make sure to provide the model path in the config.yaml 

Generation on test/data takes on the order of hours.  

If you receive GPU OOM errors here, try reducing batch size in config.

In [ ]:
config = ZaiConfig.build_from_yaml_file(os.path.join(_config_dir, "config.yaml"), experiment_id="AttNHP_zai_gen")

In [ ]:
#Max sequence length:
max_len = config.data_config.data_specs.max_len
max_len

In [ ]:
print(config.model_config)

In [ ]:
print(config.trainer_config)

In [ ]:
model_runner = ZaiTPPRunner.build_from_config(config)
model_runner.run()

In [ ]:
#Pull the predictions from the most recent model generation run (alternatively, assign `predfile` to a specific asset).
filenames = os.listdir(_checkpoint_dir)
filenames.sort(key=lambda f: os.path.getmtime(os.path.join(_checkpoint_dir, f)))
preds_glob = os.path.join(_checkpoint_dir, filenames[-1], "eval", "preds*.pkl")
predfiles = glob(preds_glob)
if len(predfiles) > 0:
    predfiles.sort(key=lambda f: os.path.getmtime(f))
    preds_fn = predfiles[-1]
    print(f"Predictions located at {preds_fn}")
else:
    print(f"No prediction files found at {preds_glob}.")

In [ ]:
with open(preds_fn, "rb") as fobj:
    preds = pickle.load(fobj)
list(preds.keys()), (len(preds['pred'][0]), len(preds['label'][0]))

## Loss curves etc.

In [ ]:
event_histories = defaultdict(list)
valid_event_histories = defaultdict(list)
event_dir = os.path.join(os.path.dirname(config.model_config.pretrained_model_dir), "../tfb_train")
valid_event_dir = os.path.join(os.path.dirname(config.model_config.pretrained_model_dir), "../tfb_valid")
event_files = os.listdir(event_dir)
valid_event_files = os.listdir(valid_event_dir)
events = []
valid_events = []
if len(event_files) > 0:
    for ef in event_files:
        for event in EventFileLoader(os.path.join(event_dir, ef)).Load():
            events.append(event)
            try:
                tag = event.summary.value[0].tag 
                event_histories[tag].append(event.summary.value[0].tensor.float_val[0])
            except Exception:
                pass    
    for ef in valid_event_files:
        for event in EventFileLoader(os.path.join(valid_event_dir, ef)).Load():
            valid_events.append(event)
            try:
                tag = event.summary.value[0].tag 
                valid_event_histories[tag].append(event.summary.value[0].tensor.float_val[0])
            except Exception:
                pass

In [ ]:
colors = list(mcd.TABLEAU_COLORS.keys())
print(f"Events from {os.path.join(*os.path.abspath(event_dir).split('/')[-3:])}")
fig, axs = plt.subplots(2, 2, figsize=(9, 6))
axs = axs.ravel()
fig.sca(axs[0])
plt.plot(range(len(event_histories["loglike"])), -1*np.array(event_histories["loglike"]), label="training loss")
plt.plot(range(len(valid_event_histories["loglike"])), -1*np.array(valid_event_histories["loglike"]), label="validation loss")

axs[0].set_title("Training loss curves")
axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Loss (-loglike)")
plt.legend()

fig.sca(axs[1])
plt.plot(range(len(valid_event_histories["acc"])), np.array(valid_event_histories["acc"]), label="accuracy", color=colors[3])
plt.plot(range(len(valid_event_histories["diff_ratio"])), np.array(valid_event_histories["diff_ratio"]), label="diff ratio", color=colors[4])

axs[1].set_title("Metrics")
axs[1].set_xlabel("Epoch")
axs[1].set_ylabel("Metric")
axs[1].set_ylim([0., 0.3])
plt.legend()

fig.sca(axs[2])
plt.plot(range(len(valid_event_histories["rmse"])), np.array(valid_event_histories["rmse"]), label="RMSE in time domain", color=colors[5])

axs[2].set_title("RMSE")
axs[2].set_xlabel("Epoch")
axs[2].set_ylabel("RMSE")
axs[2].set_ylim([2.5, None])
plt.legend()
plt.tight_layout()

# Patient-level analysis

In [ ]:
# Load raw test sequences as well.
with gzip.open("test/data/synth.train.json.gz", "rt", encoding="utf-8") as fobj:
    raw_seqs = [json.loads(x) for x in fobj.readlines()]

## Processing example: Candidates with at least one condition of interest in the forecasting region that did _not_ appear in the preceding history.

In [ ]:
long_candidates = [i for i, s in enumerate(raw_seqs) if contains_coi_not_in_prelookback(s, enc2thingname)]
len(long_candidates)

In [ ]:
# Unnormalized
intensities = preds['pred'][1]
# Normalized over conditions
normed_intensities = preds['pred'][1] / np.sum(preds['pred'][1], axis=-1, keepdims=True)
intensities.shape

In [ ]:
# Take the condition with the highest intensity as a predicted outcome.

argmax_preds = np.argmax(normed_intensities, axis=-1)

##If predictions were generated without raw intensities, do this instead:
## argmax_preds = preds['pred'][1]

In [ ]:
for x in argmax_preds[:1]:
    for y in x:
        print(y)

In [ ]:
pred_type_counter = Counter([int(a) for b in argmax_preds[:] for a in b])

In [ ]:
Counter([int(a) for b in argmax_preds[:] for a in b])

In [ ]:
Counter([int(a) for b in argmax_preds[:] for a in b])

In [ ]:
enc2thingname[39]

In [ ]:
enc2thingname[56]

In [ ]:
raw_seqs[0].keys()

In [ ]:
len(raw_seqs)

In [ ]:
len([a for b in [x['type_event'][-6:] if x['seqlen'] >= 6 else [] for x in raw_seqs] for a in b])

In [ ]:
raw_seqs_type_counter = Counter([a for b in [x['type_event'][-6:] if x['seqlen'] >= 6 else [] for x in raw_seqs] for a in b])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(11, 5))
fig.sca(ax)
# thingnames = [x[0] for x in arranged_thingnames]
raw_counts = np.array([raw_seqs_type_counter[x] for x in range(len(thingnames))])
counts = raw_counts/np.sum(raw_counts)
plt.bar(range(len(thingnames)), counts, label="Actual (synthetic)")
ax.set_xlim(0, 64)
ax.set_ylim(0, 0.5)
ax.set_xlabel("Condition")
ax.set_ylabel("Density")
ax.set_title("Predicted vs. actual (synthetic) conditions in backtesting window (6 slots)")
raw_counts = np.array([pred_type_counter[x] for x in range(len(thingnames))])
counts = raw_counts/np.sum(raw_counts)
plt.bar(range(len(thingnames)), counts, label="Predicted", alpha=0.5)
ax.legend()

In [ ]:
argmax_preds

In [ ]:
oi_in_preds = []
argmax_ranked_conditions = []
for x in argmax_preds:
    oi_in_preds.append(len({y for y in x if is_outcome_of_interest(enc2thingname.get(y, False))}))
    conds = []
    for i, y in enumerate(x):
        if is_outcome_of_interest(enc2thingname.get(y, False)):
            conds.append([(enc2thingname[y], 1.)])
        else:
            conds.append([])
#    if all([len(z) == 0 for z in conds]):
    argmax_ranked_conditions.append(conds)

In [ ]:
# Indices of patients with at least one condition of interest in predictions:
oi_in_pred_locs = np.where(np.array(oi_in_preds) > 0)
len(oi_in_pred_locs[0])

In [ ]:
oi_in_labels = []
new_oi_in_labels = []
for i, x in enumerate(preds['label'][1]):
    oi_in_labels.append(len({y for y in x if is_outcome_of_interest(enc2thingname.get(y, False))}))
    new_oi_in_labels.append(len({y for y in x if contains_coi_not_in_prelookback(raw_seqs[i], enc2thingname)}))

In [ ]:
# Total predicted conditions of interest:
sum(oi_in_preds)

In [ ]:
# Total actual conditions of interest in backtested region:
sum(oi_in_labels)

In [ ]:
# Total predicted conditions of interest with no precedent in patient history:
sum(new_oi_in_labels)

In [ ]:
# Indices of patients with predicted conditions of interest AND actual conditions of interest which only appear in the backtesting period.
interesting_locs = sorted(set(np.where(np.array(new_oi_in_labels) > 0)[0].tolist()).intersection(set(oi_in_pred_locs[0].tolist())))
len(interesting_locs)

## Patient timeline visualization

In [ ]:
# example history and predictions for synthetic patient number 37
i = interesting_locs[37]
fig, axs = plt.subplots(1, 2, figsize=(15, 5), layout="constrained", width_ratios=[3, 1])
ax = patient_timeline(raw_seqs[i], (preds['pred'][0][i], argmax_ranked_conditions[i]), axs[0], enc2thingname, window=(date(2020, 1, 1), None))
fig.sca(axs[1])
ax = patient_piechart(raw_seqs[i], axs[1], enc2thingname)
plt.show()

## Intensity scores may have different distributions in each sequence position. In addition to the argmax method, we can also generate predictions by thresholding the normalized intensities at specified percentiles.

In [ ]:
z_scores_master = zscore(intensities, axis=1)
pos_z_scores = np.array([z_scores_master[:, i, :].reshape([-1]) for i in range(z_scores_master.shape[1])])
pos_z_scores = [x[~np.isnan(x) & ~np.isinf(x)] for x in pos_z_scores]

z_scores_master.shape

In [ ]:
# example 95th percentile per position
pos_scores = normed_intensities
percentiles_by_position = [95, 95, 95, 95, 95, 95]
thresholds = [np.nanpercentile(pos_scores[:, i, :].reshape([-1]), p) for i, p in enumerate(percentiles_by_position)]

In [ ]:
colors = list(mcd.TABLEAU_COLORS.keys())
#fig, axs = plt.subplots(1, 2, figsize=(16, 5))
#fig.sca(axs[0])
plt.hist(pos_scores[:, 0, :].reshape([-1]), bins=1000, alpha=0.4, color=colors[0], label="Position 0")
axs[0].vlines([thresholds[0]], [0], [1.e7], color=colors[0])
plt.hist(pos_scores[:, 1, :].reshape([-1]), bins=1000, alpha=0.4, color=colors[1], label="Position 1")
axs[0].vlines([thresholds[1]], [0], [1.e7], color=colors[1])
plt.hist(pos_scores[:, 2, :].reshape([-2]), bins=1000, alpha=0.4, color=colors[2], label="Position 2")
axs[0].vlines([thresholds[2]], [0], [1.e7], color=colors[2])
plt.hist(pos_scores[:, 3, :].reshape([-1]), bins=1000, alpha=0.4, color=colors[3], label="Position 3")
axs[0].vlines([thresholds[3]], [0], [1.e7], color=colors[3])

axs[0].set_xlim([0, 1.])
axs[0].set_ylim([0.000, 20000])
axs[0].set_title("Distribution of scores by sequence position")
plt.legend()

In [ ]:
def top_n_predicted_conditions(patient_i, pos, n=3, min_threshold=0., lookback=6):
    scores = pos_scores[patient_i, pos]
    ordering = np.argsort(scores)[::-1]
    top_n = [(enc2thingname.get(y, f"<padding {y}>"), score) for y, score in zip(ordering, scores[ordering]) 
             if (scores[y] > min_threshold) and
                (is_outcome_of_interest(enc2thingname.get(y, False))) and 
                (y not in raw_seqs[patient_i]['type_event'][:max_len][:-lookback])][:n]
    return top_n

In [ ]:
top_n_predicted_conditions(long_candidates[5], 0, n=3, min_threshold=0)

In [ ]:
def ranked_conditions_by_position(patient_i):
    return [top_n_predicted_conditions(patient_i, k, min_threshold=thresholds[k], n=1) for k in range(pos_scores.shape[1])]

In [ ]:
all_ranked_conditions = [ranked_conditions_by_position(i) for i in tqdm(range(intensities.shape[0]))]

In [ ]:
# example history and predictions for synthetic patient number 46000 with ranked conditions based on percentiles
i = 46000

fig, axs = plt.subplots(1, 2, figsize=(15, 5), layout="constrained", width_ratios=[3, 1])
ax = patient_timeline(raw_seqs[i], (preds['pred'][0][i], all_ranked_conditions[i]), axs[0], enc2thingname, window=(None, None))
fig.sca(axs[1])
ax = patient_piechart(raw_seqs[i], axs[1], enc2thingname)
plt.show()

# Population-level analysis

In [ ]:
c = Counter()
c_patient = Counter()
for i, s in enumerate(raw_seqs):
    for x in s["type_event"][:max_len]:
        if is_outcome_of_interest(enc2thingname[x]):
            c[enc2thingname[x]] += 1
    for x in set(s["type_event"][:max_len]):
        if is_outcome_of_interest(enc2thingname[x]):
            c_patient[enc2thingname[x]] += 1


In [ ]:
inc_df = pd.DataFrame(c.values(), index=c.keys(), columns=["test events"])

In [ ]:
inc_df["test patients"] = [c_patient[x] for x in inc_df.index]

In [ ]:
inc_df["pop incidence"] = inc_df["test patients"]/len(raw_seqs)

In [ ]:
inc_df.sort_values("pop incidence", ascending=False)[:10]

In [ ]:
# Number of patients with a certain condition anywhere, example Acute Myocardial Infarction:
patients_presenting_condition = [i for i, x in enumerate(preds['label'][1]) if thingname2enc["Acute Myocardial Infarction"] in x]
len(patients_presenting_condition)